# 16 — SU vs refusal axis leverage readout (Llama 3.3 70B)

<!-- EVAL_AWARENESS_STRIPPED_v1 -->

**Question.** When SU steering jailbreaks the model, *how much* does the residual stream actually move on the refusal feature axis — compared to a refusal-direction push that lands the same compliance rate? This notebook is the mechanism half of the cross-model 'more with less' story.

We forward-pass AdvBench-30 (and a held-out neutral set) under each steering condition with `ResidualSteerer` installed (no generation), capture the post-block residual at each layer, and project onto:

1. The **refusal axis** (Arditi mean-diff direction) — does SU push refusal off baseline? By how much per σ vs the refusal-direction sweep?
2. The **SU axis itself** — sanity check that steering does push along its own direction; calibrates the leverage-curve slope for SU.

**Why this is load-bearing for the paper.** The behavioural sweep (nb15) shows SU jailbreaks at lower σ with less degenerate output. But that's a behavioural claim. The leverage curve here turns it into a *mechanistic* claim: **for equal compliance, SU moves the refusal feature less than refusal-direction steering does** (= 'less feature movement on the refusal axis', the second of the four 'more-with-less' quantities).

**Conditions.** Baseline + `SU −0.5σ` (no jailbreak) + `SU −0.65σ` (cliff edge) + `SU −0.7σ` (peak) + `SU −0.85σ` (post-cliff) + `refusal −0.4σ` (refusal-axis jailbreak peak) + `refusal +0.3σ` (defense booster). Same 7 conditions as Qwen for direct cross-model comparison.

**Self-contained / Colab-portable.** Same clone-and-load pattern as nb15. ~30 prompts × 7 conditions × 2 pools = ~420 forward passes (no generation). ~30 min on 3× A100 80GB.

**Inputs needed.**
- `exp06_lamma/directions.npz` — S/U PCA directions (steerable + readable).
- `exp_directions_llama33_70b_refusal_only/directions.npz` — refusal direction (steerable + readable).
- `data/advbench_harmful.json` — AdvBench-30 prompts.


> **Llama 3.3 70B port note.** Direct port of the Qwen 3.5 4B exp16, with eval-awareness stripped (refusal-only NPZ is the canonical Llama input).
>
> - Steering window: `L40..79` (last half of 80 layers; Qwen used `L16..31` of 32).
> - Mid-stack leverage window: `L60..72` (analogue of Qwen's `L22..28`).
> - The `EXP15_COMPLIANCE` dict in §13 still contains the **Qwen** compliance numbers as placeholders — replace with Llama numbers from `exp15_jailbreak_steering_llama33_70b/expB_compliance_summary.csv` after running notebook 15.
> - Inputs: `exp06_lamma/directions.npz` + `exp_directions_llama33_70b_refusal_only/directions.npz` (both force-tracked in git).


## 0 — GPU check

In [ ]:
import subprocess, torch
try:
    print(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout)
except FileNotFoundError:
    print('no nvidia-smi (CPU/MPS host?)')
if torch.cuda.is_available():
    n = torch.cuda.device_count()
    total_gb = sum(torch.cuda.get_device_properties(i).total_memory for i in range(n)) / 1e9
    print(f'{n} GPUs visible, total VRAM = {total_gb:.0f} GB')
else:
    print('no CUDA; will run on CPU/MPS — slow but correct.')

## 1 — Install dependencies

In [ ]:
!pip install -q 'transformers==4.46.3' 'accelerate>=0.33' huggingface_hub tqdm numpy pandas matplotlib

In [ ]:
!pip install transformers --ugrade 

## 1b — Clone the Mech_spoof repo (if not already on disk)

On a fresh Colab/pod we need the repo for source code, directions, and prompts. Skip this cell if you've rsynced the repo already and `MECH_SPOOF_ROOT` is set.

In [ ]:
import os
from pathlib import Path

REPO_URL = 'https://github.com/ChuloIva/Mech_spoof.git'

if os.environ.get('MECH_SPOOF_ROOT'):
    target = Path(os.environ['MECH_SPOOF_ROOT'])
elif Path('/content').exists():
    target = Path('/content/Mech_spoof')
elif Path('/workspace').exists():
    target = Path('/workspace/Mech_spoof')
else:
    target = Path.cwd() / 'Mech_spoof'

if not (target / 'src' / 'mech_spoof').exists():
    target.parent.mkdir(parents=True, exist_ok=True)
    print(f'cloning {REPO_URL} → {target}')
    !git clone --depth 1 {REPO_URL} {target}
else:
    print(f'repo already at {target} (skipping clone)')

os.environ['MECH_SPOOF_ROOT'] = str(target)

for sub in ['src/mech_spoof',
            'exp06_lamma/directions.npz',
            'exp_directions_llama33_70b_refusal_only/directions.npz',
            'data/advbench_harmful.json']:
    p = target / sub
    print(f'  {sub:<45s} {"OK" if p.exists() else "MISSING"}')

## 2 — Project root, paths, optional HF auth

In [ ]:
import os, sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import torch

PROJECT_ROOT = Path(os.environ.get('MECH_SPOOF_ROOT', '/workspace/Mech_spoof'))
if not PROJECT_ROOT.exists():
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
    print(f'falling back to PROJECT_ROOT={PROJECT_ROOT}')
assert (PROJECT_ROOT / 'src' / 'mech_spoof').exists(), f'no mech_spoof at {PROJECT_ROOT}/src — set MECH_SPOOF_ROOT'

sys.path.insert(0, str(PROJECT_ROOT / 'src'))

EXP6_NPZ    = PROJECT_ROOT / 'exp06_lamma' / 'directions.npz'
DIRS_NPZ    = PROJECT_ROOT / 'exp_directions_llama33_70b_refusal_only' / 'directions.npz'
ADVBENCH    = PROJECT_ROOT / 'data' / 'advbench_harmful.json'
OUT_DIR     = PROJECT_ROOT / 'exp16_su_eval_readout_llama33_70b'
OUT_DIR.mkdir(parents=True, exist_ok=True)

try:
    from google.colab import userdata
    try:
        os.environ.setdefault('HF_TOKEN', userdata.get('HF_TOKEN'))
    except Exception:
        pass
except Exception:
    pass

for label, p in [('exp06_npz', EXP6_NPZ),
                  ('dirs_npz', DIRS_NPZ), ('advbench', ADVBENCH)]:
    print(f'  {label:<14s} {p}  {"OK" if p.exists() else "MISSING"}')
print('  out_dir       ', OUT_DIR)

## 3 — Load Llama 3.3 70B

## 1c — Paste your Hugging Face token

Llama 3.3 70B is gated. Paste your `HF_TOKEN` between the quotes below and run this cell. The token is stored only in this kernel's environment (not written to disk). If you're on Colab and have already set `HF_TOKEN` in *Secrets*, you can leave this empty.


In [ ]:
import os
from huggingface_hub import login

# >>> Paste your HF token between the quotes <<<
HF_TOKEN_PASTE = ""  # e.g. "hf_AbCdEfGhIjKlMnOpQrStUvWxYz1234567890"

if HF_TOKEN_PASTE:
    os.environ['HF_TOKEN'] = HF_TOKEN_PASTE
if os.environ.get('HF_TOKEN') or os.environ.get('HUGGING_FACE_HUB_TOKEN'):
    login(token=os.environ.get('HF_TOKEN') or os.environ['HUGGING_FACE_HUB_TOKEN'],
          add_to_git_credential=False)
    print('logged in to Hugging Face')
else:
    raise RuntimeError('No HF token set. Paste it above (HF_TOKEN_PASTE = "hf_...") or set HF_TOKEN in env / Colab Secrets.')


In [ ]:
from mech_spoof.io import load_npz
from mech_spoof.models import load_model
from mech_spoof.probes import ResidualSteerer

# MECH_SPOOF_LLAMA33_MULTI_GPU_LOAD
# Multi-GPU loader. Pick precision below; load_model honours device_map='auto'
# so the 70B is automatically sharded across every visible GPU.
import torch
QUANT = 'bf16'  # default: full bf16 (~140 GB; fits 3x A100 80GB).
                # alternatives: '8bit' (~70 GB, fits 1x 80GB or 3x 40GB), '4bit' (~35 GB).

# Reserve ~4 GB per GPU for activations + KV cache; the rest is for weights.
n_gpus = torch.cuda.device_count()
if n_gpus == 0:
    raise RuntimeError('no CUDA GPUs visible')
per_gpu_gb = [int(torch.cuda.get_device_properties(i).total_memory / 1e9) for i in range(n_gpus)]
HEADROOM_GB = 4
max_memory = {i: f'{max(per_gpu_gb[i] - HEADROOM_GB, 4)}GiB' for i in range(n_gpus)}
max_memory['cpu'] = '32GiB'   # tiny CPU offload safety valve
print(f'visible GPUs: {n_gpus}, per-GPU memory budget: {max_memory}')

# Translate QUANT for load_model: 'bf16' means no quantization (full precision).
_quant_arg = 'none' if QUANT == 'bf16' else QUANT
loaded = load_model(
    'llama33_70b',
    quantization=_quant_arg,
    device_map='auto',
    max_memory=max_memory,
)
model, tok = loaded.hf_model, loaded.tokenizer
device = loaded.device
model.eval()
supports_thinking = getattr(loaded.template, '_supports_enable_thinking', False)
print(f'model={loaded.cfg.hf_id}  device={device}  n_layers={loaded.n_layers}  d_model={loaded.d_model}  '
      f'thinking_supported={supports_thinking}')

# Show how the 70B got sharded across GPUs (sanity check for multi-GPU loads).
from collections import Counter
_dev_counts = Counter()
for name, p in loaded.hf_model.named_parameters():
    _dev_counts[str(p.device)] += 1
print('parameter-tensor counts per device:', dict(_dev_counts))
# First param's device is what `model.generate` / `model(...)` expect inputs on.
print('first param device (input target):', next(loaded.hf_model.parameters()).device)


## 4 — Build direction registry

Two roles for the directions, both steerable AND readable:

- **`SU/exp06_pca_center`**. Same as nb15, layers 40..79, position `response_last`.
- **`refusal/pos-3`**. Same as nb15.

We compute `cos(SU, refusal)` per layer up-front; this is the geometric-overlap baseline for interpreting cross-axis movement under steering.


In [ ]:
# FIXED_ALPHA_STEERING_v1
# Fixed-α (repeng-canonical) steering — see nb15 for rationale.
STEER_LAYERS = list(range(40, 80))  # last half of llama 70B's 80 layers
POSITION_SU     = 'response_last'
POSITION_REF    = -3       # canonical Arditi position
PER_LAYER_SIGMA = False    # legacy flag, kept for manifest compatibility

def _unitize(v):
    v = v.astype(np.float32)
    return v / (np.linalg.norm(v) + 1e-8)

# --- SU (unit; readable + steerable) ---
exp6 = load_npz(EXP6_NPZ)
arrs6 = exp6  # llama bundles pca_center_dir + mm_raw in one npz
su_unit  = {l: _unitize(exp6[f'pca_center_dir__{POSITION_SU}__layer_{l:03d}']) for l in STEER_LAYERS}
su_raw   = {l: arrs6[f'mm_raw__{POSITION_SU}__layer_{l:03d}'].astype(np.float32) for l in STEER_LAYERS}
su_norms = {l: float(np.linalg.norm(su_raw[l])) for l in STEER_LAYERS}
su_dirs  = dict(su_unit)
su_sigma = 1.0

# --- refusal (unit; readable + steerable) ---
dirs_arrs = load_npz(DIRS_NPZ)
ref_unit  = {l: _unitize(dirs_arrs[f'refusal__mm_dir__pos_{POSITION_REF:+d}__layer_{l:03d}']) for l in STEER_LAYERS}
ref_raw   = {l: dirs_arrs[f'refusal__mm_raw__pos_{POSITION_REF:+d}__layer_{l:03d}'].astype(np.float32) for l in STEER_LAYERS}
ref_norms = {l: float(np.linalg.norm(ref_raw[l])) for l in STEER_LAYERS}
ref_dirs  = dict(ref_unit)
ref_sigma = 1.0

READ_LAYERS = list(range(loaded.n_layers))
ref_unit_read = ref_unit
su_unit_read  = su_unit

METHODS = {
    'SU':      {'dirs': su_dirs,  'sigma': su_sigma},
    'refusal': {'dirs': ref_dirs, 'sigma': ref_sigma},
}

def _cos(a, b):
    a = a / (np.linalg.norm(a) + 1e-8); b = b / (np.linalg.norm(b) + 1e-8)
    return float(np.dot(a, b))
cos_su_ref = {l: _cos(su_unit[l], ref_unit[l]) for l in STEER_LAYERS}

print(f'fixed-α steering. POSITION_REF={POSITION_REF}')
print(f'  SU  natural-scale norm median={np.median(list(su_norms.values())):.2f}')
print(f'  ref natural-scale norm median={np.median(list(ref_norms.values())):.2f}')
print(f'  cos(SU, refusal) median={np.median(list(cos_su_ref.values())):+.3f}')


## 5 — Forward + capture helper

For each (chat) prompt, run a single forward pass with the model's `output_hidden_states=True`. The forward hook from `ResidualSteerer` modifies the layer's output **in place from the wrapper's perspective**, so the hidden states the model returns reflect the steered residual stream. Read at the **last prompt token** (after `add_generation_prompt=True`) — the position the model would predict the next token from. This matches Arditi's standard readout convention for refusal and is the natural anchor for situational-awareness probes.

Returns `captures[L]` of shape `(N, d_model)` — one row per input prompt, one entry per layer in `READ_LAYERS`.

In [ ]:
BATCH_SIZE = 8
NORMALIZE  = True  # repeng-style: rescale h+δ back to ‖h‖ — must match exp15

PAD_ID = tok.pad_token_id or tok.eos_token_id

def render_chat(system: str, user: str) -> list[int]:
    msgs = [{'role': 'system', 'content': system}, {'role': 'user', 'content': user}]
    extra = {'enable_thinking': False} if supports_thinking else {}
    enc = tok.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, **extra)
    if hasattr(enc, 'input_ids'): enc = enc.input_ids
    elif isinstance(enc, dict):   enc = enc['input_ids']
    if hasattr(enc, 'tolist'):    enc = enc.tolist()
    if isinstance(enc, list) and enc and isinstance(enc[0], list): enc = enc[0]
    return [int(x) for x in enc]

def _left_pad_batch(seqs):
    max_len = max(len(s) for s in seqs)
    input_ids = torch.full((len(seqs), max_len), PAD_ID, dtype=torch.long)
    attn_mask = torch.zeros((len(seqs), max_len), dtype=torch.long)
    for i, s in enumerate(seqs):
        n = len(s)
        input_ids[i, max_len - n:] = torch.tensor(s, dtype=torch.long)
        attn_mask[i, max_len - n:] = 1
    return input_ids, attn_mask, max_len

@torch.no_grad()
def _forward_capture_one_batch(seqs, method, k):
    """Returns dict[layer_idx] -> np.ndarray (B, d_model) at the last prompt token."""
    input_ids, attn_mask, _ = _left_pad_batch(seqs)
    input_ids = input_ids.to(device); attn_mask = attn_mask.to(device)

    def _run():
        return model(input_ids=input_ids, attention_mask=attn_mask,
                      output_hidden_states=True, use_cache=False, return_dict=True)

    if method is None or k == 0.0:
        out = _run()
    else:
        m = METHODS[method]
        with ResidualSteerer(loaded, m['dirs'], coeff=k * m['sigma'], normalize=NORMALIZE):
            out = _run()

    # hidden_states is tuple of length n_layers+1: [embeddings, layer_0_out, ..., layer_{N-1}_out].
    # We want the post-block residual at layer L → hidden_states[L+1], at the last (= rightmost) token.
    hs = out.hidden_states
    captures = {}
    for L in READ_LAYERS:
        h = hs[L + 1][:, -1, :]   # (B, d_model)
        captures[L] = h.detach().to(torch.float32).cpu().numpy()
    return captures

def forward_capture_batch(pairs, method, k):
    """Forward-pass `pairs` of (system, user). Returns captures[L]: (N, d_model) over all prompts."""
    seqs = [render_chat(s, u) for s, u in pairs]
    out: dict[int, list] = {L: [] for L in READ_LAYERS}
    for i in range(0, len(seqs), BATCH_SIZE):
        chunk = _forward_capture_one_batch(seqs[i:i + BATCH_SIZE], method, k)
        for L in READ_LAYERS:
            out[L].append(chunk[L])
    return {L: np.concatenate(out[L], axis=0) for L in READ_LAYERS}

# Smoke test
_caps = forward_capture_batch([('You are a helpful assistant.', 'What is 2+2?')], method=None, k=0.0)
print(f'smoke: captured {len(_caps)} layers, layer 16 shape = {_caps[16].shape}')

## 6 — Condition grid

Smaller than exp15 — we don't need the full sweep, just the diagnostic points.

In [ ]:
# Sparser unified α grid for the leverage-curve readout (forward only).
# Same α set for SU and refusal so the leverage-slope comparison is apples-to-apples.
CONDITIONS = [
    ('baseline',        None,       0.0),
    ('SU α=-1.5',       'SU',      -1.5),
    ('SU α=-1.0',       'SU',      -1.0),
    ('SU α=-0.5',       'SU',      -0.5),
    ('SU α=-0.3',       'SU',      -0.3),
    ('SU α=-0.15',      'SU',      -0.15),
    ('SU α=+0.3',       'SU',      +0.3),
    ('SU α=+1.0',       'SU',      +1.0),
    ('refusal α=-1.5',  'refusal', -1.5),
    ('refusal α=-1.0',  'refusal', -1.0),
    ('refusal α=-0.5',  'refusal', -0.5),
    ('refusal α=-0.3',  'refusal', -0.3),
    ('refusal α=-0.15', 'refusal', -0.15),
    ('refusal α=+0.3',  'refusal', +0.3),
    ('refusal α=+1.0',  'refusal', +1.0),
]
for n, m, k in CONDITIONS:
    print(f'  {n:<18s}  method={m!s:<8s}  α={k:+.2f}')


## 7 — Prompt set

Two pools:

- **harmful**: AdvBench-30 (same as nb15).
- **neutral**: a small held-out set of benign requests. Controls for whether SU shifts the refusal axis on *any* input or only on harmful prompts (content gating).

Both go through the same generic helpful-assistant system prompt (matches nb15 expB).


In [ ]:
BENIGN_SYSTEM = 'You are a helpful, harmless, and honest assistant.'

harmful = json.loads(ADVBENCH.read_text())
if isinstance(harmful, dict) and 'prompts' in harmful:
    HARMFUL_PROMPTS = harmful['prompts']
elif isinstance(harmful, list):
    HARMFUL_PROMPTS = [p['prompt'] if isinstance(p, dict) and 'prompt' in p else p for p in harmful]
else:
    raise ValueError(f'unexpected advbench format: {type(harmful)}')
HARMFUL_PROMPTS = HARMFUL_PROMPTS[:30]

NEUTRAL_PROMPTS = [
    'Explain why the sky appears blue during the day.',
    'Write a haiku about morning coffee.',
    'What are the main differences between Python lists and tuples?',
    'Recommend three classic novels for someone new to literary fiction.',
    'How does a refrigerator keep food cold?',
    'Summarise the plot of Hamlet in three sentences.',
    'What is the capital of Mongolia, and what is one interesting fact about the country?',
    'Give me a quick recipe for scrambled eggs that serves two.',
    'What is the difference between weather and climate?',
    'Tell me about the history of the printing press.',
    'How do I water a fiddle-leaf fig without overdoing it?',
    'Write a polite email declining an invitation to a wedding.',
    'What does "compound interest" mean in plain English?',
    'Suggest a 30-minute beginner workout routine I can do at home.',
    'How does anaesthesia work, in broad strokes?',
]

POOLS = {
    'harmful': [(BENIGN_SYSTEM, p) for p in HARMFUL_PROMPTS],
    'neutral': [(BENIGN_SYSTEM, p) for p in NEUTRAL_PROMPTS],
}
for name, pool in POOLS.items():
    print(f'  {name:<10s}  N={len(pool):3d}')

## 8 — Sweep: capture residuals + project onto axes

Per (condition, pool, prompt, layer), record:

- `refusal_proj` = `h · refusal_unit` — the headline quantity for the leverage curve.
- `su_proj` = `h · su_unit` — sanity check that steering pushes along its own direction.
- `h_norm` = `‖h‖` — debugging; with `NORMALIZE=True`, baseline and steered should match closely.

Both projections are only available at layers 40..79 (where the unit was extracted at the matching position).


In [ ]:
from tqdm.auto import tqdm

rows = []
for cond_name, method, k in tqdm(CONDITIONS, desc='conditions'):
    for pool_name, pool in POOLS.items():
        captures = forward_capture_batch(pool, method, k)
        for L in READ_LAYERS:
            H = captures[L]                                 # (N, d)
            hn = np.linalg.norm(H, axis=-1)                 # (N,)
            if L in STEER_LAYERS:
                rf = H @ ref_unit_read[L]
                su = H @ su_unit_read[L]
            else:
                rf = np.full(H.shape[0], np.nan)
                su = np.full(H.shape[0], np.nan)
            for i in range(H.shape[0]):
                rows.append({
                    'condition': cond_name,
                    'method': method or 'none',
                    'k': k,
                    'pool': pool_name,
                    'prompt_idx': i,
                    'layer': L,
                    'refusal_proj': float(rf[i]),
                    'su_proj':      float(su[i]),
                    'h_norm':       float(hn[i]),
                })

df = pd.DataFrame(rows)
df.to_csv(OUT_DIR / 'projections_per_prompt.csv', index=False)
print(f'rows = {len(df)}, conditions = {df.condition.nunique()}, layers = {df.layer.nunique()}')
df.head()


## 9 — Aggregate per (condition, pool, layer)

In [ ]:
agg = (df.groupby(['condition', 'method', 'k', 'pool', 'layer'])
         [['refusal_proj', 'su_proj', 'h_norm']]
         .agg(['mean', 'std'])
         .reset_index())
agg.columns = ['_'.join(c).rstrip('_') for c in agg.columns]
agg.to_csv(OUT_DIR / 'projections_agg.csv', index=False)

# Δ vs baseline at the (pool, layer) level — the headline quantity.
base = (df[df.condition == 'baseline']
        .groupby(['pool', 'layer'])
        [['refusal_proj', 'su_proj']]
        .mean()
        .rename(columns={'refusal_proj': 'refusal_proj_base',
                          'su_proj': 'su_proj_base'})
        .reset_index())
delta = df.merge(base, on=['pool', 'layer'])
delta['refusal_proj_delta'] = delta['refusal_proj'] - delta['refusal_proj_base']
delta['su_proj_delta']      = delta['su_proj']      - delta['su_proj_base']

delta_agg = (delta.groupby(['condition', 'method', 'k', 'pool', 'layer'])
                  [['refusal_proj_delta', 'su_proj_delta']]
                  .agg(['mean', 'std'])
                  .reset_index())
delta_agg.columns = ['_'.join(c).rstrip('_') for c in delta_agg.columns]
delta_agg.to_csv(OUT_DIR / 'projections_delta_vs_baseline.csv', index=False)
print(f'agg rows = {len(agg)}; delta rows = {len(delta_agg)}')
agg.head()


## 10 — Plots

Three panels, all on the harmful pool (steered layers L40..79):

1. **Refusal-axis Δ vs baseline per layer** — the headline plot. Per-condition curves of `Δ(h · refusal_unit)`. The vertical separation between SU curves and the refusal-direction curves at matched compliance is the 'less feature movement on the refusal axis' claim.
2. **SU-axis Δ vs baseline per layer** — sanity check that SU steering does push along its own direction; calibrates the leverage normalisation.
3. **Refusal-axis Δ, neutral pool** — content control. If the refusal axis only moves on harmful prompts under SU, the SU push is content-gated rather than a global feature drift.


In [ ]:
import matplotlib.pyplot as plt

FIG_DIR = OUT_DIR / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

ORDER = [c for c, _, _ in CONDITIONS]
COLOURS = {
    'baseline':       '#444444',
    'SU −0.5σ':       '#1f77b4',
    'SU −0.65σ':      '#ff7f0e',
    'SU −0.7σ':       '#d62728',
    'SU −0.85σ':      '#9467bd',
    'refusal −0.4σ':  '#2ca02c',
    'refusal +0.3σ':  '#17becf',
}

def _plot_per_layer(values, ylabel, title, savepath, layers=READ_LAYERS, hl_layers=STEER_LAYERS):
    fig, ax = plt.subplots(figsize=(10, 5))
    for cond in ORDER:
        ys = [values.get((cond, L), float('nan')) for L in layers]
        ax.plot(layers, ys, marker='o', ms=3, label=cond, color=COLOURS.get(cond))
    ax.axvspan(min(hl_layers) - 0.5, max(hl_layers) + 0.5, color='#fff5d8', alpha=0.5, zorder=-1)
    ax.set_xlabel('layer'); ax.set_ylabel(ylabel); ax.set_title(title)
    ax.legend(fontsize=8, loc='best'); ax.grid(alpha=0.3)
    fig.tight_layout(); fig.savefig(savepath, dpi=130); plt.show()

def _table(metric_col, agg_df, pool):
    sub = agg_df[agg_df.pool == pool]
    return {(r['condition'], int(r['layer'])): float(r[metric_col]) for _, r in sub.iterrows()}

# Plot 1 — refusal-axis Δ, harmful pool (HEADLINE)
_plot_per_layer(
    values=_table('refusal_proj_delta_mean', delta_agg, 'harmful'),
    ylabel='Δ (h · refusal_unit)  vs baseline',
    title='Refusal-axis shift under steering — harmful prompts (HEADLINE)',
    savepath=FIG_DIR / 'refusal_axis_delta_harmful.png',
    layers=STEER_LAYERS,
)

# Plot 2 — SU-axis Δ, harmful pool (sanity)
_plot_per_layer(
    values=_table('su_proj_delta_mean', delta_agg, 'harmful'),
    ylabel='Δ (h · su_unit)  vs baseline',
    title='SU-axis shift under steering — harmful prompts (sanity)',
    savepath=FIG_DIR / 'su_axis_delta_harmful.png',
    layers=STEER_LAYERS,
)

# Plot 3 — refusal-axis Δ, neutral pool (content control)
_plot_per_layer(
    values=_table('refusal_proj_delta_mean', delta_agg, 'neutral'),
    ylabel='Δ (h · refusal_unit)  vs baseline',
    title='Refusal-axis shift under steering — NEUTRAL prompts (content control)',
    savepath=FIG_DIR / 'refusal_axis_delta_neutral.png',
    layers=STEER_LAYERS,
)


## 11 — Save manifest + summary

In [ ]:
manifest = {
    'experiment': '16_su_refusal_axis_readout',
    'model': loaded.cfg.hf_id,
    'steer_layers': STEER_LAYERS,
    'read_layers': READ_LAYERS,
    'positions': {'SU': POSITION_SU, 'refusal': POSITION_REF,
                   'read_position': 'last_prompt_token'},
    'sigma': {'SU': float(su_sigma), 'refusal': float(ref_sigma)},
    'per_layer_sigma': PER_LAYER_SIGMA,
    'normalize': NORMALIZE,
    'cos_per_layer': {
        'SU_ref': cos_su_ref,
    },
    'conditions': [{'name': n, 'method': m, 'k': k} for n, m, k in CONDITIONS],
    'pools': {name: len(pool) for name, pool in POOLS.items()},
}
(OUT_DIR / 'manifest.json').write_text(json.dumps(manifest, indent=2))
print('wrote', OUT_DIR / 'manifest.json')
print('outputs:')
for p in sorted(OUT_DIR.rglob('*')):
    if p.is_file():
        print(' ', p.relative_to(OUT_DIR))


## What to look for

**Headline claim — 'less feature movement on the refusal axis'.** At matched behavioural compliance (e.g. `SU −0.7σ` ≈ 100% comply vs `refusal −0.4σ` ≈ 87% comply), the refusal-axis Δ should be substantially *smaller* under SU steering than under refusal-direction steering. Quantified by §13's slope-of-feature-shift-per-σ — SU's slope on the refusal axis should be flatter than the refusal-direction sweep's slope.

**Sanity:** SU steering pushes along the SU axis (Plot 2 should show clean monotonic shift with k); refusal steering doesn't push on SU (cos is small).

**Content gating:** if Plot 3 (refusal axis on neutral prompts) is much flatter than Plot 1 (harmful prompts), the SU push is content-gated — only routes through the refusal feature when the prompt is harmful. Surprising result if the pattern is symmetric.

**Surprises to flag:**
- Refusal-axis Δ shifts at layers *upstream* of the steered range (L<40). Measurement artefact — no causal pathway upstream of the hook.
- `SU −0.5σ` (no jailbreak in nb15) already shows large refusal-axis shift. Means the refusal feature moves freely below the behavioural threshold; feature movement is necessary but not sufficient for compliance.
- `refusal −0.4σ` jailbreaks in nb15 but with *less* refusal-axis Δ than `SU −0.7σ`. Would invert the headline claim — flag immediately.


## 12 — Mid-stack focus

Zoomed view of refusal-axis Δ at L60..72 (mid + late stack), dropping the L73-79 amplification tail so the y-axis isn't dominated by the late-stack spike. This is the slice §13 averages over for the leverage-curve summary.


In [ ]:
MID_LAYERS = list(range(40, 73))  # mid + late stack on llama 70B; drops L73-79 amplification tail

# Refusal-axis Δ, harmful pool, MID-STACK ONLY
_plot_per_layer(
    values=_table('refusal_proj_delta_mean', delta_agg, 'harmful'),
    ylabel='Δ (h · refusal_unit)  vs baseline',
    title='Refusal-axis shift — harmful prompts (mid-stack L40-72)',
    savepath=FIG_DIR / 'refusal_axis_delta_harmful_midstack.png',
    layers=MID_LAYERS,
    hl_layers=MID_LAYERS,
)

# Refusal-axis Δ, neutral pool, MID-STACK ONLY (content control)
_plot_per_layer(
    values=_table('refusal_proj_delta_mean', delta_agg, 'neutral'),
    ylabel='Δ (h · refusal_unit)  vs baseline',
    title='Refusal-axis shift — neutral prompts (mid-stack L40-72)',
    savepath=FIG_DIR / 'refusal_axis_delta_neutral_midstack.png',
    layers=MID_LAYERS,
    hl_layers=MID_LAYERS,
)

# Summary table at L66 (representative mid-stack layer) for the writeup
print('\nSummary at L66 (representative mid-stack):')
print(f'{"condition":<16s}  {"refusal_Δ_harm":>14s}  {"refusal_Δ_neut":>14s}  '
      f'{"content_gated":>13s}  {"su_Δ_harm":>10s}')
for cond, _, _ in CONDITIONS:
    rh = _table('refusal_proj_delta_mean', delta_agg, 'harmful').get((cond, 66), float('nan'))
    rn = _table('refusal_proj_delta_mean', delta_agg, 'neutral').get((cond, 66), float('nan'))
    sh = _table('su_proj_delta_mean',      delta_agg, 'harmful').get((cond, 66), float('nan'))
    print(f'{cond:<16s}  {rh:>+14.3f}  {rn:>+14.3f}  {rh-rn:>+13.3f}  {sh:>+10.3f}')


## 13 — Per-σ leverage curves

For each method, plot mid-stack mean refusal-axis Δ vs steering coefficient `k`, with nb15 compliance overlaid as marker size. The slope of `refusal_Δ` vs `k` quantifies the **per-σ leverage** of each attack on the refusal feature. The slope ratio (refusal-method ÷ SU-method) is the headline 'less feature movement per unit jailbreak' number for the paper.

Compliance numbers below are still pasted from Qwen exp15 — replace with the Llama numbers from `exp15_jailbreak_steering_llama33_70b/expB_compliance_summary.csv` after running notebook 15 on Llama.


In [ ]:
# Compliance from exp15 (PLACEHOLDER: Qwen numbers — replace with llama numbers post-nb15 run)
EXP15_COMPLIANCE = {
    'baseline':       0.00,
    'SU −0.5σ':       0.00,
    'SU −0.65σ':      0.80,
    'SU −0.7σ':       1.00,
    'SU −0.85σ':      0.90,
    'refusal −0.4σ':  0.87,
    'refusal +0.3σ':  0.00,
}

# Mid-stack mean feature shift per condition (avg over L60..72 — past the steering-injection layers,
# before the L73-79 amplification kicks in)
LEVERAGE_LAYERS = list(range(60, 73))  # mid-stack on llama 70B (analogue of Qwen 4B L22-28)

def _midstack_mean(metric_col, agg_df, pool, condition):
    sub = agg_df[(agg_df.pool == pool) & (agg_df.condition == condition)]
    sub = sub[sub.layer.isin(LEVERAGE_LAYERS)]
    return float(sub[metric_col].mean())

# Build per-condition summary table
import math
summary = []
for cond, method, k in CONDITIONS:
    summary.append({
        'condition': cond,
        'method':    method or 'baseline',
        'k':         k,
        'refusal_Δ_mid':  _midstack_mean('refusal_proj_delta_mean', delta_agg, 'harmful', cond),
        'su_Δ_mid':       _midstack_mean('su_proj_delta_mean',      delta_agg, 'harmful', cond),
        'compliance':     EXP15_COMPLIANCE.get(cond, math.nan),
    })
summary_df = pd.DataFrame(summary)
summary_df.to_csv(OUT_DIR / 'leverage_summary.csv', index=False)
print('Mid-stack mean (L60–72):')
print(summary_df.to_string(index=False))

# --- Leverage plot: refusal-axis Δ vs k, separated by method ---
fig, ax = plt.subplots(figsize=(8, 5.5))

for method, marker, colour in [('SU', 'o', '#d62728'), ('refusal', 's', '#2ca02c')]:
    sub = summary_df[summary_df.method == method].sort_values('k')
    if len(sub) == 0: continue
    sizes = 20 + 380 * sub['compliance'].fillna(0).values
    ax.scatter(sub.k, sub['refusal_Δ_mid'], s=sizes, alpha=0.75, marker=marker,
               color=colour, edgecolor='black', linewidth=0.5,
               label=f'{method}  (size = compliance)')
    ax.plot(sub.k, sub['refusal_Δ_mid'], '--', color=colour, alpha=0.4, lw=1)
    if len(sub) >= 2:
        slope, intercept = np.polyfit(sub.k.values, sub['refusal_Δ_mid'].values, 1)
        xs = np.linspace(sub.k.min(), sub.k.max(), 20)
        ax.plot(xs, slope * xs + intercept, '-', color=colour, lw=1.2, alpha=0.6)
        ax.text(sub.k.iloc[-1], sub['refusal_Δ_mid'].iloc[-1],
                f'  slope={slope:+.2f}', color=colour, fontsize=9, va='center')

base_row = summary_df[summary_df.method == 'baseline']
if len(base_row):
    ax.scatter(base_row.k, base_row['refusal_Δ_mid'], s=40, marker='x', color='black', label='baseline')
ax.axhline(0, color='#888', lw=0.5); ax.axvline(0, color='#888', lw=0.5)
ax.set_xlabel('steering coefficient k  (× σ)')
ax.set_ylabel('Δ (h · refusal_unit)  mid-stack mean')
ax.set_title('Refusal-axis leverage: feature shift per σ (Llama 3.3 70B)')
ax.legend(fontsize=9, loc='best'); ax.grid(alpha=0.3)

fig.tight_layout()
fig.savefig(FIG_DIR / 'leverage_curves.png', dpi=130)
plt.show()

# Slope ratios — quantify the leverage advantage
print('\nLeverage (slope of refusal-axis shift per unit k):')
slopes = {}
for method in ['SU', 'refusal']:
    sub = summary_df[summary_df.method == method].sort_values('k')
    if len(sub) >= 2:
        slope, _ = np.polyfit(sub.k.values, sub['refusal_Δ_mid'].values, 1)
        slopes[method] = slope
        print(f'  {method:<8s}  slope = {slope:+.3f}  per σ')

if 'SU' in slopes and 'refusal' in slopes and slopes['SU'] != 0:
    ratio = abs(slopes['refusal'] / slopes['SU'])
    print(f'\n  refusal / SU slope ratio = {ratio:.2f}x  '
          f'(refusal-direction steering moves the refusal feature {ratio:.1f}× more per σ than SU)')

# Compliance-per-feature efficiency: at the jailbreak peak, how much refusal_Δ does
# each method spend per percentage point of compliance gained?
print('\nFeature-spend efficiency at jailbreak peaks:')
for cond in ['SU −0.7σ', 'refusal −0.4σ']:
    sub = summary_df[summary_df.condition == cond]
    if len(sub) == 0: continue
    row = sub.iloc[0]
    if row.compliance > 0:
        eff = row['refusal_Δ_mid'] / row.compliance
        print(f'  {cond:<16s}  refusal_Δ = {row["refusal_Δ_mid"]:+.2f}  '
              f'compliance = {row.compliance:.2%}  '
              f'(refusal_Δ per unit compliance = {eff:+.2f})')
